In [0]:
# ============================================================
# FASE 2B — Transform Gold: agregaciones para el dashboard
# ============================================================
from pyspark.sql.functions import (
    col, count, round as spark_round,
    avg, sum as spark_sum, rank
)
from pyspark.sql.window import Window

# ----------------------------------------------------------
# 1. Cargar datos desde Silver
# ----------------------------------------------------------
df_silver = spark.table("etl_cine.peliculas_italianas_silver")
print(f"Registros en silver: {df_silver.count()}")

# ----------------------------------------------------------
# 2. Tabla Gold 1: Ranking de popularidad
# ----------------------------------------------------------
window_ranking = Window.orderBy(col("popularidad").desc())

df_ranking = df_silver \
    .withColumn("ranking", rank().over(window_ranking)) \
    .select(
        "ranking",
        "titulo",
        "titulo_original",
        "anio_estreno",
        "generos",
        "popularidad",
        "valoracion_media",
        "num_votos"
    ) \
    .orderBy("ranking")

df_ranking.show(10, truncate=False)

df_ranking.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("etl_cine.gold_ranking_popularidad")

print("✓ Tabla guardada: etl_cine.gold_ranking_popularidad")

# ----------------------------------------------------------
# 3. Tabla Gold 2: Evolución por año
# ----------------------------------------------------------
df_por_anio = df_silver \
    .groupBy("anio_estreno") \
    .agg(
        count("pelicula_id").alias("num_peliculas"),
        spark_round(avg("valoracion_media"), 2).alias("valoracion_media"),
        spark_round(avg("popularidad"), 2).alias("popularidad_media"),
        spark_round(avg("num_votos"), 0).alias("votos_medios")
    ) \
    .orderBy("anio_estreno")

df_por_anio.show(20, truncate=False)

df_por_anio.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("etl_cine.gold_evolucion_por_anio")

print("✓ Tabla guardada: etl_cine.gold_evolucion_por_anio")